<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/example_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

In [ ]:
#!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
#%cd mask-predict

In [ ]:
#!pip install -e .

In [1]:
from explain import *
import pandas as pd
import json

In [2]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

maskedToken = 'slave'
#clusterName = 'slave'
modelName = "Livingwithmachines/bert_1760_1900"
dataPath = 'masking_data'
processedFolder = 'processed_data'
predCol = "pred_bert_1760_1900"
iloc_range = (0, -1)

In this notebook we analyse the newspaper sentences where machine in mentioned by BLERT predicts words related to slavery.

In [3]:
#!unzip -o "{collection}_{maskedToken}_clusters{genre_suffix}.tsv.zip"

We rank sentences by their similarity to the 'slave' cluster based on the BLERT predictions. 

In [4]:

data_df = pd.read_csv(f"{dataPath}/{collection}_{maskedToken}_clusters{genre_suffix}.tsv", sep="\t", index_col=0)
#data_df = data_df.sort_values(by=f"{clusterName}_1760_1900", ascending=False).reset_index(drop=True)
data_df.head()

,article_path,identifier,date,shelfmarks,publisher,title,edition,contributors,creators,prevSentence,...,slave_1760_1900,artisan_1760_1900,woman_1760_1900,machine_contemporary,boy_contemporary,girl_contemporary,slave_contemporary,artisan_contemporary,woman_contemporary,genre
0,001484048_01_text.json,1484048,1854,British Library HMNTS 10027.e.11.,NaN,"The Jordan and the Rhine: or, the East and the...",NaN,NaN,"GRAHAM, William - D.D., M.R.I.A","In every sense of the word , Syria is the most...",...,0.219,0.226,0.295,0.095,0.238,0.238,0.150,0.149,0.236,Non-fiction
1,003924165_01_text.json,3924165,1875,British Library HMNTS 9602.df.1.,NaN,The Annals of Kansas,NaN,NaN,"WILDER, Daniel W.",""" Sec 13 .",...,0.991,0.427,0.378,0.230,0.307,0.303,0.971,0.395,0.356,Non-fiction
2,001042371_01_text.json,1042371,1890,British Library HMNTS 012631.k.27.,R. Bentley & Sons,The Parting of the Ways. A novel,NaN,NaN,"EDWARDS, Matilda Barbara Betham.",It has nothing to do with coals or blankets .,...,0.207,0.215,0.268,0.189,0.226,0.240,0.207,0.175,0.263,Fiction
3,002536859_01_text.json,2536859,1835,British Library HMNTS 1047.i.10./British Libra...,NaN,"Ten Years in South Africa, including a particu...",NaN,NaN,"MOODIE, John Wedderburn Dunbar.",The slaves are peculiarly lowered in the scale...,...,0.878,0.344,0.351,0.171,0.240,0.220,0.476,0.231,0.271,Non-fiction
4,003588968_05_text.json,3588968,1870,British Library HMNTS 9504.h.1.,NaN,"The Family History of England, civil, military...",NaN,NaN,"TAYLOR, James - D.D., of Glasgow",No further dealings or demands were inter pose...,...,0.563,0.258,0.249,0.183,0.270,0.240,0.730,0.314,0.302,Non-fiction


We get the first 1000 sentences, and the top 5 predictions. This means we only look at context words where our target words of interest (like 'slave' appear in the top five words).

In [5]:
texts, predicted_targets = build_texts_targets(
    data_df, start=iloc_range[0], end=iloc_range[1],
    pred_col=predCol, top_n=5
)

We run the explainer using BLERT as the model we inspect, our targets are the top 5 predictions, the texts are the 1000 sentences selected in the previous cell.

# Compute Integrated Gradients.

In [29]:
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
results = explainer.explain(texts, predicted_targets, word_agg="max")

In [ ]:
with open(f'results_{collection}_{maskedToken}_{clusterName}_pred.json', 'w') as f:
    json.dump(results, f)

Only retain results for context word contribution when slave or machine are predicted by BLERT.

In [ ]:
results_df = result_as_dataframe(results, ['slave','machine'])
results_df.to_csv(f'results_{collection}_{maskedToken}_{clusterName}_pred.csv')

# Contrastive Analysis

Contrastive analysis, we use slave and machine as target words for all sentences, ignoring the actual predictions.

In [ ]:
if maskedToken == 'slave':
      contrastive_dict = {'slave':'machine', 'slaves':'machines'}
elif maskedToken == 'machine':
    contrastive_dict = {'machine':'slave', 'machines':'slaves'}

data_df['targetExpressionProccessed'] = data_df['targetExpression'].apply(lambda x: x.lower().strip())
data_df['targetContrExpressionProccessed'] = data_df['targetExpressionProccessed'].apply(lambda x: contrastive_dict.get(x, x))
contrastive_targets = data_df.iloc[iloc_range[0]:iloc_range[1]][['targetContrExpressionProccessed', 'targetExpressionProccessed']].values.tolist()

len(texts) == len(contrastive_targets)

True

In [11]:
contrastive_targets[:10]

[['machine', 'slave'],
 ['machines', 'slaves'],
 ['machines', 'slaves'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machines', 'slaves']]

In [ ]:
results2 = explainer.explain(texts, contrastive_targets, word_agg="max")

In [ ]:
with open(f'results_{collection}_{maskedToken}_constrastive.json', 'w') as f:
    json.dump(results2, f)

In [7]:
with open(f'{processedFolder}/results_{collection}_{maskedToken}_constrastive.json', 'r') as f:
    results2 = json.load(f)

In [13]:
unique = list(set(x for sublist in contrastive_targets for x in sublist))
unique

['machine', 'slave', 'slaves', 'machines']

In [ ]:
results_df = result_as_dataframe(results2,unique)
results_df.shape

  8%|▊         | 4114/53726 [00:49<07:07, 116.15it/s] 

In [ ]:
results_df.to_csv(f'results_{collection}_{maskedToken}_constrastive.csv')

KeyboardInterrupt: 

# Load data

In [ ]:
resultType = 'constrastive' # or 'manual' | 'pred' | 'constrastive'
results_df = pd.read_csv(f'{processedFolder}/results_{collection}_{maskedToken}_{resultType}.csv', index_col=0)
results_df.shape

(95320, 4)

In [19]:
slave_df = results_df[results_df['Target'] == 'slave'].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))
machine_df = results_df[results_df['Target'] == 'machine'].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))

In [20]:
machine_df.shape, slave_df.shape

((6041, 3), (6041, 3))

In [21]:
slave_df['diff_slave_machine'] = slave_df['avg'] - machine_df['avg']

In [22]:
#slave_df['count'].plot(kind='hist', bins=100, title='Count of tokens in slave cluster')

In [24]:
slave_df[slave_df['count'] > 3].sort_values('avg', ascending=False).head(20)[['avg', 'count', 'std']]

,avg,count,std
Token,,,
slave,0.359436,4,0.078727
ex,0.315547,4,0.357644
figues,0.311125,4,0.160180
taxing,0.304839,6,0.187585
prisoner,0.303414,7,0.262567
machine,0.295893,152,0.240835
press,0.271841,23,0.175835
workers,0.264780,4,0.252133
federal,0.262693,7,0.291454


In [45]:
results_df[results_df['Token'] == 'workers'].id.unique()

array([ 27,  28, 103, 695])

In [53]:
sentence = ' '.join(results_df[results_df.Target=='slave'][results_df.id==27].Token.values)
print(sentence)
highlight_context_tokens(explainer, sentence, target="machine", word_agg="max")

/var/folders/d2/ydv0grbd38985h6_95t0vdjw0000gp/T/ipykernel_46723/213404804.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  sentence = ' '.join(results_df[results_df.Target=='slave'][results_df.id==27].Token.values)


thirty federal [MASK] workers at portsmouth , virginia , have gone over to the confederates .


'\n    <div id="tokviz_3684f04ad7e0461cb572cec9e8d166ec">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>machine</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.215111\' style=\'background:rgba(30, 136, 229, 0.425); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>thirty</span> <span class=\'tok\' data-score=\'0.379976\' style=\'background:rgba(30, 136, 229, 0.674); padding:2px 4px; m

In [56]:
slave_df[slave_df['count'] > 3].sort_values('diff_slave_machine', ascending=False).head(20)[['diff_slave_machine','count']]

,diff_slave_machine,count
Token,,
passive,0.283737,4
motive,0.280113,11
workers,0.210754,4
manufacturers,0.194624,9
continental,0.155886,4
ex,0.149111,4
sweep,0.125914,5
letter,0.108669,4
majority,0.097175,5


In [106]:
results_df[results_df['Token'] == 'passive'].id.unique()

array([120, 208, 218, 332])

In [108]:
sentence = ' '.join(results_df[results_df.Target=='slave'][results_df.id==120].Token.values)
print(sentence)
highlight_context_tokens(explainer, sentence, target="machine", word_agg="max")

/var/folders/d2/ydv0grbd38985h6_95t0vdjw0000gp/T/ipykernel_46723/3341190476.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  sentence = ' '.join(results_df[results_df.Target=='slave'][results_df.id==120].Token.values)


the modern soldier most no longer remain a passive [MASK] , but must be trained as a skilled labourer .


'\n    <div id="tokviz_a52b4e124774473b9fce0b0b5204d9d8">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>machine</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.247684\' style=\'background:rgba(30, 136, 229, 0.530); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>the</span> <span class=\'tok\' data-score=\'0.241764\' style=\'background:rgba(30, 136, 229, 0.520); padding:2px 4px; marg

In [13]:
sentence = data_df.iloc[0]['maskedSentence']
highlight_context_tokens(explainer, sentence, target="slave", word_agg="max")

'\n    <div id="tokviz_1fddd69b1bd24353b316146d215b3bc6">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>slave</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.037372\' style=\'background:rgba(30, 136, 229, 0.155); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>technical</span> <span class=\'tok\' data-score=\'0.038193\' style=\'background:rgba(30, 136, 229, 0.156); padding:2px 4px; 

In [84]:
df_slave = results_df[results_df['Target'] == 'slave'].reset_index(drop=True)
df_machine = results_df[results_df['Target'] == 'machine'].reset_index(drop=True)

In [85]:
df_slave

,Token,Score,Target,id
0,technical,0.037370,slave,0
1,skill,0.038191,slave,0
2,and,0.057329,slave,0
3,exact,0.002623,slave,0
4,work,0.106219,slave,0
...,...,...,...,...
47655,country,0.638198,slave,999
47656,is,0.168336,slave,999
47657,socially,-0.001073,slave,999
47658,happy,-0.033089,slave,999


In [90]:
# individual instances where the difference is highest
df_slave['diff_slave_machine'] = df_slave["Score"].sub(df_machine["Score"], fill_value=0)
df_slave.sort_values(by = 'diff_slave_machine',ascending=False).head(20)

,Token,Score,Target,id,diff_slave_machine
15731,manufacturers,0.759577,slave,328,1.171528
14174,motive,0.382474,slave,296,0.999608
35448,",",0.401962,slave,711,0.794930
12612,motive,0.049255,slave,264,0.687638
12298,motive,-0.009037,slave,258,0.653092
28064,•,0.412836,slave,575,0.647596
42362,.,0.542174,slave,879,0.596667
42085,.,0.542174,slave,870,0.596667
4034,machines,0.677359,slave,83,0.586673
4051,machines,0.677359,slave,84,0.586673


In [97]:
sentence = ' '.join(results_df[results_df.Target=='slave'][results_df.id==264].Token.values)
highlight_context_tokens(explainer, sentence, target="machine", word_agg="max")

/var/folders/d2/ydv0grbd38985h6_95t0vdjw0000gp/T/ipykernel_46723/639573581.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  sentence = ' '.join(results_df[results_df.Target=='slave'][results_df.id==264].Token.values)


'\n    <div id="tokviz_5da4ce5870a44d7c8259330f1463b63f">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>machine</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.048731\' style=\'background:rgba(30, 136, 229, 0.157); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>all</span> <span class=\'tok\' data-score=\'0.155001\' style=\'background:rgba(30, 136, 229, 0.282); padding:2px 4px; marg

In [102]:
slave_df_sent = results_df[results_df['Target'] == 'slave'].groupby('id').agg(avg=('Score', 'mean'), std=('Score', 'std'))
machine_df_sent = results_df[results_df['Target'] == 'machine'].groupby('id').agg(avg=('Score', 'mean'), std=('Score', 'std'))
slave_df_sent['diff_slave_machine'] = slave_df_sent['avg'] - machine_df_sent['avg']
slave_df_sent.sort_values('diff_slave_machine', ascending=False).head(20)[['diff_slave_machine']].sort_values('diff_slave_machine', ascending=False).head(20)[['diff_slave_machine']]

,diff_slave_machine
id,
38,0.082978
193,0.077232
238,0.076054
63,0.056526
397,0.056259
902,0.056247
44,0.051641
351,0.051359
823,0.050648


In [105]:
sentence = ' '.join(results_df[results_df.Target=='slave'][results_df.id==193].Token.values)
highlight_context_tokens(explainer, sentence, target="machine", word_agg="max")

/var/folders/d2/ydv0grbd38985h6_95t0vdjw0000gp/T/ipykernel_46723/2252826002.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  sentence = ' '.join(results_df[results_df.Target=='slave'][results_df.id==193].Token.values)


'\n    <div id="tokviz_554c03f4e170451ca5b13664d56966f1">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>machine</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'0.005281\' style=\'background:rgba(30, 136, 229, 0.116); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>-</span> <span class=\'tok\' data-score=\'0.144609\' style=\'background:rgba(30, 136, 229, 0.548); padding:2px 4px; margin

# Experimental code

Use clustering for interpretation. Results are not really satisfying.

In [ ]:
token_embeddings_df, clustered_results_df, cluster_summary_df, centroid_df = build_token_cluster_summary(
    results_df,
    model_name=modelName,
    n_clusters=20,
    score_column="Score",
    batch_size=64,
    device="cuda",
)

cluster_summary_df.head()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


: 

In [63]:
cluster_summary_df = pd.read_csv(f'{processedFolder}/cluster_summary_{collection}_{maskedToken}_{clusterName}_{resultType}.csv', index_col=0)
cluster_summary_df.head()

,cluster,Target,token_count,row_count,avg_score,top_tokens
0,0,machine,1151,1715,0.076754,"['sanderson', 'rissington', 'leeming', 'weedon..."
1,0,slave,1151,1715,0.054716,"['sanderson', 'rissington', 'leeming', 'weedon..."
2,1,machine,1,2,0.046368,['ra']
3,1,slave,1,2,0.035996,['ra']
4,2,machine,238,3216,0.029806,"['willing', 'secure', 'partial', 'comfortable'..."


In [11]:
token_embeddings_df = pd.read_json(f'{processedFolder}/token_embeddings_{collection}_{maskedToken}_{clusterName}_{resultType}.json')
token_embeddings_df.head()

,Token,cluster,embedding,emb_x,emb_y
0,technical,7,"[0.6489257812, 0.1229248047, 0.6889648438, 0.1...",1.812875,0.045541
1,skill,7,"[0.5151367188, 0.0305175781, 0.1949462891, 0.2...",1.446464,-0.397857
2,and,7,"[0.2863769531, 0.0814819336, 0.1817626953, 0.0...",1.845062,-0.286311
3,exact,3,"[0.1801757812, 0.1121826172, 0.2131347656, 0.0...",-0.463441,-3.330458
4,work,7,"[0.7412109375, 0.1424560547, 0.1431884766, 0.0...",1.340968,0.203639


In [12]:
fig = plot_token_embeddings_interactive(token_embeddings_df)
fig.show()


: 